# Drywall QA — CLIPSeg Training
Runs on Colab T4 via VS Code Colab extension.

In [ ]:
# Cell 1: Setup
!pip install -q torch torchvision transformers albumentations opencv-python-headless tqdm

import os
if not os.path.isdir('drywall'):
    !git clone https://github.com/Aaryan2304/drywall.git
%cd drywall
print('Ready')

In [ ]:
# Cell 2: Generate instance masks (skip if already done)
!python -m src.data.mask_conversion --data_root . --subsets cracks/train cracks/valid cracks/test taping/train taping/valid taping/test

In [ ]:
# Cell 3: Verify data
import pathlib, json
for ds in ['cracks', 'taping']:
    for split in ['train', 'valid', 'test']:
        img_dir = pathlib.Path(ds) / split
        ann_file = img_dir / '_annotations.coco.json'
        ann = json.load(open(ann_file))
        masks = list((img_dir / 'instance_masks').glob('*.png')) if (img_dir / 'instance_masks').exists() else []
        print(f'{ds}/{split}: {len(ann["images"])} images, {len(ann["annotations"])} annotations, {len(masks)} masks')

In [ ]:
# Cell 4: Train
# batch_size=2, grad_accum=2 is fine for T4 (16GB VRAM)
!python train.py --batch_size 2 --grad_accum 2 --max_steps 2500

In [ ]:
# Cell 5: Save results to Google Drive (optional)
# If you want to copy outputs off Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r outputs /content/drive/MyDrive/drywall_outputs
# Or just download outputs/ from the Colab file browser

In [ ]:
# Cell 6: Quick sanity check — run inference on 5 test images
!python infer.py --checkpoint outputs/checkpoints/best.pt --subset test --max_images 5 --out_dir outputs/sanity_test

In [ ]:
# Cell 7: Run full evaluation
!python evaluate.py --checkpoint outputs/checkpoints/best.pt --out_dir outputs/eval_results